In [3]:
import findspark
import pyspark
from pyspark.sql import SparkSession

In [4]:
"""
Exercise #43 -
"""

'\nExercise #43 -\n'

In [5]:
findspark.init()
sc = pyspark.SparkContext.getOrCreate()
spark = SparkSession.builder.getOrCreate()

In [6]:
"""
RDD SOLUTION
"""

'\nRDD SOLUTION\n'

In [7]:
threshold = 5

In [8]:
readingsRDD = sc.textFile("data/readings.txt").map(lambda x: (x.split(",")[0], int(x.split(",")[5])))
readingsRDD.collect()

[('s1', 4),
 ('s2', 4),
 ('s3', 3),
 ('s4', 2),
 ('s5', 6),
 ('s1', 5),
 ('s2', 3),
 ('s3', 2),
 ('s4', 1),
 ('s5', 7),
 ('s1', 6),
 ('s3', 5),
 ('s4', 1),
 ('s5', 2),
 ('s1', 5),
 ('s2', 3),
 ('s3', 4),
 ('s5', 4),
 ('s5', 4),
 ('s1', 0),
 ('s2', 0),
 ('s3', 0)]

In [9]:
# Part 3
mappedRDD = readingsRDD.map(lambda x: (x[0], (1 if x[1] < threshold else 0, 1)))
criticalityRDD = mappedRDD.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])).map(lambda x: (x[0], x[1][0] / x[1][1]))
criticalityRDD.collect()

[('s1', 0.4), ('s3', 0.8), ('s5', 0.6), ('s2', 1.0), ('s4', 1.0)]

In [10]:
# Part 4
filteredCriticalStationsRDD = criticalityRDD.filter(lambda x: x[1] > 0.8).sortBy(lambda x: x[1], ascending=False)
filteredCriticalStationsRDD.collect()

[('s2', 1.0), ('s4', 1.0)]

In [11]:
# Part 5
def define_timeslot(x):
    if x < 4:
        return '[0-3]'
    elif x < 8:
        return '[4-7]'
    elif x < 12:
        return '[8-11]'
    elif x < 16:
        return '[12-15]'
    elif x < 20:
        return '[16-19]'
    else:
        return '[20-23]'

In [12]:
readingsRDD = sc.textFile("data/readings.txt").map(lambda x: (x.split(",")[0], int(x.split(",")[2]), float(x.split(",")[5])))
readingsRDD.collect()

[('s1', 0, 4.0),
 ('s2', 0, 4.0),
 ('s3', 0, 3.0),
 ('s4', 0, 2.0),
 ('s5', 0, 6.0),
 ('s1', 0, 5.0),
 ('s2', 0, 3.0),
 ('s3', 0, 2.0),
 ('s4', 0, 1.0),
 ('s5', 0, 7.0),
 ('s1', 15, 6.0),
 ('s3', 15, 5.0),
 ('s4', 15, 1.0),
 ('s5', 15, 2.0),
 ('s1', 15, 5.0),
 ('s2', 15, 3.0),
 ('s3', 15, 4.0),
 ('s5', 15, 4.0),
 ('s5', 0, 4.0),
 ('s1', 0, 0.0),
 ('s2', 0, 0.0),
 ('s3', 0, 0.0)]

In [13]:
mappedRDD = readingsRDD.map(lambda x: (x[0] + ',' + define_timeslot(x[1]), (1 if x[2] < threshold else 0, 1)))
mappedRDD.collect()

[('s1,[0-3]', (1, 1)),
 ('s2,[0-3]', (1, 1)),
 ('s3,[0-3]', (1, 1)),
 ('s4,[0-3]', (1, 1)),
 ('s5,[0-3]', (0, 1)),
 ('s1,[0-3]', (0, 1)),
 ('s2,[0-3]', (1, 1)),
 ('s3,[0-3]', (1, 1)),
 ('s4,[0-3]', (1, 1)),
 ('s5,[0-3]', (0, 1)),
 ('s1,[12-15]', (0, 1)),
 ('s3,[12-15]', (0, 1)),
 ('s4,[12-15]', (1, 1)),
 ('s5,[12-15]', (1, 1)),
 ('s1,[12-15]', (0, 1)),
 ('s2,[12-15]', (1, 1)),
 ('s3,[12-15]', (1, 1)),
 ('s5,[12-15]', (1, 1)),
 ('s5,[0-3]', (1, 1)),
 ('s1,[0-3]', (1, 1)),
 ('s2,[0-3]', (1, 1)),
 ('s3,[0-3]', (1, 1))]

In [14]:
criticalityRDD = mappedRDD.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])).map(lambda x: (x[0], x[1][0] / x[1][1]))
criticalityRDD.collect()

[('s2,[0-3]', 1.0),
 ('s3,[0-3]', 1.0),
 ('s4,[0-3]', 1.0),
 ('s1,[12-15]', 0.0),
 ('s4,[12-15]', 1.0),
 ('s1,[0-3]', 0.6666666666666666),
 ('s5,[0-3]', 0.3333333333333333),
 ('s3,[12-15]', 0.5),
 ('s5,[12-15]', 1.0),
 ('s2,[12-15]', 1.0)]

In [16]:
filteredCriticalStationsRDD = criticalityRDD.filter(lambda x: x[1] > 0.8).sortBy(lambda x: x[1], ascending=False)
filteredCriticalStationsRDD.collect()
#filteredCriticalStationsRDD.saveAsTextFile("critical_stations_timeslots.txt")

[('s2,[0-3]', 1.0),
 ('s3,[0-3]', 1.0),
 ('s4,[0-3]', 1.0),
 ('s4,[12-15]', 1.0),
 ('s5,[12-15]', 1.0),
 ('s2,[12-15]', 1.0)]

In [17]:
# Part 7
neighborsRDD = sc.textFile("data/neighbors.txt").map(lambda x: x.split(","))
neighborsRDD.collect()

[['s1', 's2 s3'], ['s2', 's1 s5'], ['s3', 's1'], ['s4', 's5'], ['s5', 's4 s2']]

In [18]:
mappedRDD = sc.textFile("data/readings.txt").map(lambda x: (x.split(",")[0], (define_timeslot(int(x.split(",")[2])), float(x.split(",")[5]))))
mappedRDD.collect()


[('s1', ('[0-3]', 4.0)),
 ('s2', ('[0-3]', 4.0)),
 ('s3', ('[0-3]', 3.0)),
 ('s4', ('[0-3]', 2.0)),
 ('s5', ('[0-3]', 6.0)),
 ('s1', ('[0-3]', 5.0)),
 ('s2', ('[0-3]', 3.0)),
 ('s3', ('[0-3]', 2.0)),
 ('s4', ('[0-3]', 1.0)),
 ('s5', ('[0-3]', 7.0)),
 ('s1', ('[12-15]', 6.0)),
 ('s3', ('[12-15]', 5.0)),
 ('s4', ('[12-15]', 1.0)),
 ('s5', ('[12-15]', 2.0)),
 ('s1', ('[12-15]', 5.0)),
 ('s2', ('[12-15]', 3.0)),
 ('s3', ('[12-15]', 4.0)),
 ('s5', ('[12-15]', 4.0)),
 ('s5', ('[0-3]', 4.0)),
 ('s1', ('[0-3]', 0.0)),
 ('s2', ('[0-3]', 0.0)),
 ('s3', ('[0-3]', 0.0))]

In [19]:
filteredRDD = mappedRDD.filter(lambda x: x[1][1] == 0)
filteredRDD.collect()

[('s1', ('[0-3]', 0.0)), ('s2', ('[0-3]', 0.0)), ('s3', ('[0-3]', 0.0))]

In [30]:
def neighbors_full(readings, neighbors):
    flag = False
    sensors_list = []
    for sensor in readings:
        sensors_list.append(sensor[0])
    for line in readings:
        for n in neighbors:
            if line[0] == n[0]:
                for neighbor in n[1].split(" "):
                    if neighbor not in sensors_list:
                        flag = False
                        break
                    else:
                        flag = True
                break
        yield line, flag

In [32]:
finalRDD = sc.parallelize(neighbors_full(filteredRDD.collect(), neighborsRDD.collect())).filter(lambda x: x[1] == True).map(lambda x: x[0])
finalRDD.collect()

[('s1', ('[0-3]', 0.0)), ('s3', ('[0-3]', 0.0))]

In [181]:
"""
SPARKSQL SOLUTION
"""

'\nSPARKSQL SOLUTION\n'